In [1]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp
from functools import partial

print(jax.devices())
print(len(jax.devices()))

[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
8


In [2]:
import time

# Create our mesh! We're running on a TPU v2-8 4x2 slice with names 'X' and 'Y'.
assert len(jax.devices()) == 8
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('X', 'Y'))

# A little utility function to help define our sharding. A PartitionSpec is our
# sharding (a mapping from axes to names).
def P(*args):
  return jax.NamedSharding(mesh, jax.sharding.PartitionSpec(*args))

# We shard both A and B over the non-contracting dimension and A over the contracting dim.
A = jnp.zeros((8, 2048), dtype=jnp.bfloat16, device=P('X', 'Y'))
B = jnp.zeros((2048, 8192), dtype=jnp.bfloat16, device=P(None, 'Y'))

# We can perform a matmul on these sharded arrays! out_shardings tells us how we want
# the output to be sharded. JAX/XLA handles the rest of the sharding for us.
y = jax.jit(lambda A, B: jnp.einsum('BD,DF->BF', A, B), out_shardings=P('X', 'Y'))(A, B)

In [17]:
def bench(name, f, x, iters=20):
  # compile, warmup
  y = f(x)
  y.block_until_ready()

  start = time.time()
  for _ in range(iters):
    y = f(x)
    y.block_until_ready()
  end = time.time()

  print(f"{name}: {(end - start) / iters * 1e6:.1f} us")
  return y

In [18]:
x_ag = jnp.ones(
    (8, 1024, 1024),
    dtype=jnp.bfloat16,
)

def allgather_pmap(x):
    return jax.lax.all_gather(x, axis_name="i")

allgather_pmap = jax.pmap(
    allgather_pmap,
    axis_name="i",
)

y_ag = bench("AllGather pmap", allgather_pmap, x_ag)
print("AllGather output shape:", y_ag.shape)

AllGather pmap: 174320.0 us
AllGather output shape: (8, 8, 1024, 1024)


In [19]:
x_ar = jnp.ones(
    (8, 1024, 1024),
    dtype=jnp.bfloat16,
)

def allreduce_pmap(x):
    return jax.lax.psum(x, axis_name="i")

allreduce_pmap = jax.pmap(
    allreduce_pmap,
    axis_name="i",
)

y_ar = bench("AllReduce pmap", allreduce_pmap, x_ar)
print("AllReduce output shape:", y_ar.shape)

AllReduce pmap: 55596.5 us
AllReduce output shape: (8, 1024, 1024)


In [20]:
x_rs = jnp.ones(
    (8, 8, 1024, 1024),
    dtype=jnp.bfloat16,
)

def reducescatter_pmap(x):
    return jax.lax.psum_scatter(
        x,
        axis_name="i",
        scatter_dimension=0,
    )

reducescatter_pmap = jax.pmap(
    reducescatter_pmap,
    axis_name="i",
)

y_rs = bench("ReduceScatter pmap", reducescatter_pmap, x_rs)
print("ReduceScatter output shape:", y_rs.shape)

ReduceScatter pmap: 258764.3 us
ReduceScatter output shape: (8, 1024, 1024)


In [21]:
x_a2a = jnp.ones(
    (8, 8, 1024, 1024),
    dtype=jnp.bfloat16,
)

def alltoall_pmap(x):
    return jax.lax.all_to_all(
        x,
        axis_name="i",
        split_axis=0,
        concat_axis=0,
    )

alltoall_pmap = jax.pmap(
    alltoall_pmap,
    axis_name="i",
)

y_a2a = bench("AllToAll pmap", alltoall_pmap, x_a2a)
print("AllToAll output shape:", y_a2a.shape)

AllToAll pmap: 387023.6 us
AllToAll output shape: (8, 8, 1024, 1024)
